# `groundinsight` — persistence (JSON & SQLite)

Round-trip checks for the two persistence paths supported by the
package:

- **JSON** via `gi.save_network_to_json` / `gi.load_network_from_json`
  (Pydantic `model_dump_json` under the hood).
- **SQLite** via `gi.save_network_to_db` / `gi.load_network_from_db`
  (SQLAlchemy ORM, with `BusType` / `BranchType` table sharing).

**Plausibility checks**
1. Bus, branch, fault and source counts survive the round trip.
2. The same `run_fault` call on the loaded network produces the same
   reduction factor as on the original.
3. `BusType` / `BranchType` saved separately can be reloaded and
   reused.


In [1]:
import sys
import os
# Make the src/ tree importable when running from the notebooks/ folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import groundinsight as gi
from groundinsight.models.core_models import BusType, BranchType, ComplexNumber

print('groundinsight', gi.__version__)


groundinsight 0.3.0


In [2]:
def make_bus_type():
    """Unit bus impedance so the shield path dominates."""
    return BusType(
        name='BusUnit',
        description='Unit-like bus impedance for plausibility tests',
        system_type='Grounded',
        voltage_level=20.0,
        impedance_formula='rho * 0 + 1.0 + I * f * 0',
    )

def make_ms_cable():
    """MV cable with the reference impedances above."""
    return BranchType(
        name='MSCable',
        description='MV cable reference branch',
        grounding_conductor=True,
        self_impedance_formula='(rho * 0 + 0.25 + I * 0.6)*l',
        mutual_impedance_formula='(rho * 0 + 0.0 + I * 0.6)*l',
    )

def make_ohl():
    """Overhead line without shield."""
    return BranchType(
        name='OHLine',
        description='Overhead line without shield',
        grounding_conductor=False,
        self_impedance_formula='NaN',
        mutual_impedance_formula='NaN',
    )


## 1. Build a small reference network

In [3]:
def build_reference():
    net = gi.create_network(name='Persistence', frequencies=[50, 250])
    bus_type = make_bus_type()
    cable = make_ms_cable()
    for i in range(1, 5):
        gi.create_bus(name=f'bus{i}', type=bus_type, network=net)
    for i in range(1, 4):
        gi.create_branch(name=f'b{i}{i+1}', type=cable,
                         from_bus=f'bus{i}', to_bus=f'bus{i+1}',
                         length=1.0, network=net)
    gi.create_source(name='src', bus='bus1',
                     values={50: 100.0, 250: 60.0}, network=net)
    gi.create_fault(name='fault', bus='bus4',
                    scalings={50: 1.0, 250: 1.0}, network=net)
    return net

net = build_reference()
gi.run_fault(net, fault_name='fault')
r_orig = net.results['fault'].reduction_factor.value[50.0]
print(f'original network: {len(net.buses)} buses, {len(net.branches)} branches')
print(f'r at 50 Hz = {r_orig:.6f}')


original network: 4 buses, 3 branches
r at 50 Hz = 0.384615


## 2. JSON round-trip

In [4]:
json_path = 'persistence.json'
gi.save_network_to_json(net, json_path)

net_json = gi.load_network_from_json(json_path)
print(f'loaded JSON: {len(net_json.buses)} buses, {len(net_json.branches)} branches')

gi.create_paths(network=net_json)
gi.run_fault(net_json, fault_name='fault')
r_json = net_json.results['fault'].reduction_factor.value[50.0]
print(f'r after JSON round-trip = {r_json:.6f}   delta = {abs(r_json - r_orig):.2e}')

assert len(net_json.buses)    == len(net.buses)
assert len(net_json.branches) == len(net.branches)
assert abs(r_json - r_orig) < 1e-9
print('JSON round-trip ok.')


loaded JSON: 4 buses, 3 branches
r after JSON round-trip = 0.384615   delta = 0.00e+00
JSON round-trip ok.


## 3. SQLite round-trip

The SQLite path additionally exercises the `BusTypeDB` /
`BranchTypeDB` association tables.


In [5]:
db_path = 'persistence.db'
# Start clean: remove old DB file if present
import os
if os.path.exists(db_path):
    os.remove(db_path)

gi.start_dbsession(db_path)
gi.save_network_to_db(net, overwrite=True)

net_db = gi.load_network_from_db(name=net.name)
print(f'loaded DB: {len(net_db.buses)} buses, {len(net_db.branches)} branches')

gi.create_paths(network=net_db)
gi.run_fault(net_db, fault_name='fault')
r_db = net_db.results['fault'].reduction_factor.value[50.0]
print(f'r after SQLite round-trip = {r_db:.6f}   delta = {abs(r_db - r_orig):.2e}')

assert len(net_db.buses)    == len(net.buses)
assert len(net_db.branches) == len(net.branches)
assert abs(r_db - r_orig) < 1e-9
print('SQLite round-trip ok.')


Database session started with 'persistence.db'.
loaded DB: 4 buses, 3 branches
r after SQLite round-trip = 0.384615   delta = 5.55e-17
SQLite round-trip ok.


## 4. BusType / BranchType save/load standalone

In [6]:
bt = BusType(
    name='RoundtripBus',
    description='dummy bus type',
    system_type='Grounded',
    voltage_level=20.0,
    impedance_formula='rho / 100 + I * f / 50',
)
gi.save_bustype_to_db(bt, overwrite=True)
loaded = gi.load_bustypes_from_db()
print('BusTypes in DB:', list(loaded.keys()))
assert 'RoundtripBus' in loaded
assert loaded['RoundtripBus'].impedance_formula == bt.impedance_formula

br = BranchType(
    name='RoundtripBranch',
    description='dummy branch type',
    grounding_conductor=False,
    self_impedance_formula='NaN',
    mutual_impedance_formula='NaN',
)
gi.save_branchtype_to_db(br, overwrite=True)
loaded_br = gi.load_branchtypes_from_db()
print('BranchTypes in DB:', list(loaded_br.keys()))
assert 'RoundtripBranch' in loaded_br

gi.close_dbsession()
print('standalone type round-trip ok.')


BusTypes in DB: ['BusUnit', 'RoundtripBus']
BranchTypes in DB: ['MSCable', 'RoundtripBranch']
Database session closed.
standalone type round-trip ok.


## 5. Cleanup

In [7]:
for path in (json_path, db_path):
    if os.path.exists(path):
        os.remove(path)
        print(f'removed {path}')


removed persistence.json
removed persistence.db


---

**Notes**

- The JSON path uses Pydantic's native `model_dump_json` /
  `model_validate_json`, so any model change must keep the JSON layer
  in mind.
- The SQLAlchemy path stores `frequencies` as `PickleType` and
  impedance dicts as JSON with string keys (see CLAUDE.md). The
  round-trip check in this notebook would catch a regression in
  either layer.
